# FASE 3 · SCRIPT 1 — DIAGNÓSTICO DE LA MUESTRA

 
Parte de lo que ya existe y no lo vuelve a calcular:
  - **coronas**: con log área, log H/L y log elongación. Esta capa es la muestra de la Fase 3.
  - **rectángulos**: los eventos con la información de las Fases 0 y 1
    (fuente, tipo de movimiento, medidas del rectángulo).
 
**Qué hace** 
  1. Verifica que la capa de coronas es la muestra esperada (n, IDs, métricas
     completas, evento en revisión ausente) y muestra si los eventos con pocas
     celdas están en ella y dónde caen.
  2. Tabla tipo de movimiento × fuente (caídas incluidas, sin cambios).
  3. Clases de geología, geomorfología y cobertura en la corona de cada evento,
     con el área de cada clase en la capa. Es un diagnóstico para decidir el
     agrupamiento; la extracción definitiva (clase mayoritaria en el buffer)
     va en el siguiente script.
  4. Tamaño típico de los eventos, para fijar el radio del buffer.
 
Salidas (en CARPETA_SALIDA):
  * `F3_S1_reporte.txt`          todo lo que se imprime en consola
  * `F3_S1_pocas_celdas.csv`     eventos con pocas celdas: si están en la muestra y dónde
  * `F3_S1_tipo_x_fuente.csv`    tabla de contingencia
  * `F3_S1_clases_<capa>.csv`    clases por condicionante categórico
  * `F3_S1_tamano_eventos.csv`   percentiles de ancho, semiancho, largo y radio equivalente
  * `F3_S1_muestra.csv`          muestra con todo lo anterior (base para el script 2)


# Librerías y configuración

### **Qué hace**

Carga las librerías y reúne las capas, los campos y los valores esperados.



In [65]:
# Arman rutas, revisan números y fechan el reporte
import os
import math
import datetime

# Manejo de datos y tablas
import numpy as np
import pandas as pd
 
try:
    import arcpy
except ImportError:      # permite probar las funciones de tablas fuera de ArcGIS
    arcpy = None
 
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
 
ENC = "utf-8-sig"        # Permite que los CSV abran correctamente en Excel o R


# ============================================================================
# CONFIGURACIÓN 
# ============================================================================
 
# Capas 

CORONAS        = "coronas"        # muestra de la Fase 2
RECTANGULOS    = "fase1_movilidad"    # información de las Fases 0 y 1


# Condicionantes categóricos   

GEOLOGIA       = "Geologia"
COBERTURA      =   "cobertura_amva" 
GEOMORFOLOGIA  = "Geomorfologia"



 
# --- Campos en CORONAS ------------------------------------------------------
CAMPO_ID_CORONAS = "ID_POL"

CAMPO_GRUPO      = "grupo"          # "Mapeo sobre imágenes" / "Institucional"
CAMPO_TIPO       = "tipo_mov"       # tipo de movimiento usado en la Fase 2

CAMPO_EXCLUIDO   = None

# --- Campos en RECTANGULOS --------------------------------------------------
CAMPO_ID_RECT = "ID_evento"

# Para cada métrica: nombre del campo y si ya está en log10 (True) o en valor
# normal (False). Si está en valor normal, el script calcula el log10.
CAMPO_AREA  = "LOG_AREA";     AREA_EN_LOG  = True      # área en m² o su log10
CAMPO_HL    = "LOG_H_L";       HL_EN_LOG    = True
CAMPO_ELONG = "LOG_E";    ELONG_EN_LOG = True
CAMPO_ANCHO = "MBG_Width"     # ancho del rectángulo (m)
CAMPO_LARGO = "MBG_Length"    # largo del rectángulo (m)


#  Campo con la clase en cada capa categórica

CAMPO_CLASE_GEOLOGIA      = "NOMBRE"
CAMPO_CLASE_GEOMORFOLOGIA = "NOMBRE"
CAMPO_CLASE_COBERTURA     = "NOMBRE"   
 
#  Eventos especiales 
IDS_POCAS_CELDAS = []    # IDs de los eventos con pocas celdas de la Fase 1
IDS_EN_REVISION  = []    # evento con recorte en revisión (no debería estar en la muestra)
N_ESPERADO       = 441
N_RECTANGULOS    = 453   # eventos en la capa de rectángulos (inventario completo)




CARPETA_SALIDA = r"D:\Users\tasal\Desktop\Tesis de grado\fase3" #Carpeta a la que van todos los resultados

# FUNCIONES (no dependen de arcpy)

### **Qué hace**

En este bloque se definen las tablas del diagnóstico. 

Cada función se encarga de un detalle particular 

|Función|Descripción|
|---|---|
|`normalizar_id(v)`|convierte identificadores y códigos en texto comparable|
|`codigo_raster(x)`|Lee los raster (en caso de tener), es **None** si es el valor sin dato (−9999) |
|`a_log10(serie, ya_en_log)`| Calcula el $log_{10}$ en caso de que la métrica no estuviera ya en logaritmo |
|`verificar_muestra()`|arma la tabla de verificación, con estado `OK`, `REVISAR` o `info`. Revisa el número de eventos, los identificadores repetidos, los eventos sin rectángulo, las métricas vacías y los valores del grupo. También comprueba que la elongación de la **Fase 1** sea el lado largo entre el lado corto del rectángulo: compara `LOG_ELONG` con  $log_{10}\text{(largo/ancho)}$ y cuenta las diferencias mayores que 0,01.|
|`aplicar_dominio()`|Se activa en caso de que un campo use un dominio de valores codificados y cambia cada código por su descripción |
|`tabla_traslapes()`|lista las coronas que caen en más de un polígono de una capa y si esos polígonos tienen clases distintas.|
|`tabla_pocas_celdas()`|Dice si cada evento de una lista está en la muestra y dónde cae.|
|`tabla_tipo_grupo(m)`|tabla de contingencia de tipo de movimiento por fuente|
|`tabla_clases(m, col, areas_km2)`|eventos por clase y fuente, porcentaje de la muestra y área de la clase en la capa. Incluye las clases sin eventos|
|`medidas_tamano(m)`|el semiancho es la mitad del ancho y el radio equivalente es el de un círculo con la misma área del evento $\sqrt{A/\pi}$|
|`resumen_tamano(m)`| percentiles 10, 25, 50, 75 y 90 de las medidas, para todos los eventos y por fuente. |





In [66]:

class Reporte:
    """Imprime en consola y guarda todo en un archivo de texto."""
 
    def __init__(self, ruta):
        self.ruta = ruta
        self.lineas = []
 
    def __call__(self, texto=""):
        print(texto)
        self.lineas.append(str(texto))
 
    def titulo(self, texto):
        self("")
        self("=" * 78)
        self(texto)
        self("=" * 78)
 
    def tabla(self, df, index=False):
        self(df.to_string(index=index))
 
    def aviso(self, texto):
        self("  AVISO: " + texto)
 
    def guardar(self):
        with open(self.ruta, "w", encoding="utf-8") as f:
            f.write("\n".join(self.lineas))
 
 
def normalizar_id(v):
    """Texto comparable para IDs y códigos (12, 12.0 y '12' -> '12')."""
    if v is None:
        return None
    if isinstance(v, float):
        if math.isnan(v):
            return None
        if v.is_integer():
            return str(int(v))
    s = str(v).strip()
    return s if s else None
 
 
def codigo_raster(x):
    """Código leído de un ráster, o None si es NoData (-9999)."""
    try:
        xf = float(x)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(xf) or xf <= -9998:
        return None
    return normalizar_id(xf)
 
 
def a_log10(serie, ya_en_log):
    """Devuelve el log10 de la métrica; si ya viene en log10, la deja igual.
    Valores vacíos, no numéricos o <= 0 quedan como NaN (y la verificación los cuenta)."""
    x = pd.to_numeric(serie, errors="coerce")
    if ya_en_log:
        return x
    return np.log10(x.where(x > 0))
 
 
def verificar_muestra(m, n_coronas, excluidos, n_esperado, n_rect, dup_rect, n_rect_esperado, ids_revision):
    """Controles sobre la muestra (coronas no excluidas, unidas a sus rectángulos).
    excluidos: DataFrame con ID y EXCLUIDO (motivo) de las coronas que no entran."""
    metricas = ["LOG_AREA", "LOG_HL", "LOG_ELONG"]
    sin_rect = int((~m["_CON_RECT"]).sum())
    fuera = n_rect - (len(m) - sin_rect)          # rectángulos que no entran en la muestra
    incompletas = int(m[metricas].isna().any(axis=1).sum())
    duplicados = int(m["ID"].duplicated().sum()) + int(m["ID"].isna().sum())
    en_rev = sorted(set(m["ID"]) & set(ids_revision))
    texto_excl = "; ".join(f"{i} ({mot})" for i, mot in zip(excluidos["ID"], excluidos["EXCLUIDO"]))
 
    # Coherencia: la elongación de la Fase 1 debe ser largo/ancho del mismo rectángulo
    w = pd.to_numeric(m["ANCHO_R"], errors="coerce")
    l = pd.to_numeric(m["LARGO_R"], errors="coerce")
    log_e_rect = np.log10(np.maximum(w, l) / np.minimum(w, l))
    dif = (m["LOG_ELONG"].abs() - log_e_rect).abs().dropna()
    n_dif = int((dif > 0.01).sum())
    texto_dif = (f"{float(dif.max()):.4f}; eventos con dif. > 0,01: {n_dif}" if len(dif)
                 else "sin datos para comparar")
 
    conteo_grupos = m["GRUPO"].value_counts()
    sin_grupo = int(m["GRUPO"].isna().sum())
    texto_grupos = ", ".join(f"{g} ({n})" for g, n in conteo_grupos.items())
    if sin_grupo:
        texto_grupos += f", vacío ({sin_grupo})"
 
    filas = [
        ("Eventos en la capa de coronas", str(n_coronas), None),
        ("Excluidos (motivo)", f"{len(excluidos)}: {texto_excl}" if len(excluidos) else "0", None),
        ("Eventos en la muestra", f"{len(m)} (esperados {n_esperado})", len(m) == n_esperado),
        ("IDs vacíos o repetidos en la muestra", str(duplicados), duplicados == 0),
        ("Eventos en la capa de rectángulos", f"{n_rect} (esperados {n_rect_esperado})", n_rect == n_rect_esperado),
        ("IDs vacíos o repetidos en rectángulos", str(dup_rect), dup_rect == 0),
        ("Eventos de la muestra sin su rectángulo", str(sin_rect), sin_rect == 0),
        ("Rectángulos que quedan fuera de la muestra", f"{fuera} (esperados {n_rect_esperado - n_esperado})",
         fuera == n_rect_esperado - n_esperado),
        ("Eventos con alguna de las tres métricas vacía", str(incompletas), incompletas == 0),
        ("Elongación vs. MBG_Length/MBG_Width (dif. máx. en log10)", texto_dif, len(dif) > 0 and n_dif == 0),
        ("Valores del campo grupo", texto_grupos, len(conteo_grupos) == 2 and sin_grupo == 0),
        ("Evento en revisión presente en la muestra", ", ".join(en_rev) if en_rev else "no", not en_rev),
    ]
    estado = {True: "OK", False: "REVISAR", None: "info"}
    return pd.DataFrame([{"Verificación": a, "Resultado": b, "Estado": estado[c]} for a, b, c in filas])
 
 
def aplicar_dominio(clases, areas, traslapes, dominio):
    """Cambia los códigos de un dominio por su descripción en las clases, las áreas
    y los traslapes. Códigos que no estén en el dominio se dejan como están."""
    def t(c):
        return dominio.get(c, c) if c is not None else None
    clases = {i: t(c) for i, c in clases.items()}
    traslapes = {i: [t(c) for c in l] for i, l in traslapes.items()}
    if areas:
        nuevas = {}
        for c, a in areas.items():
            nuevas[t(c)] = nuevas.get(t(c), 0.0) + a
        areas = nuevas
    return clases, areas, traslapes
 
 
def tabla_traslapes(traslapes, ids_muestra):
    """Coronas de la muestra que caen en más de un polígono de una capa categórica."""
    filas = [{"ID": i, "Polígonos": len(l), "Clases": " | ".join(l),
              "Clases distintas": "sí" if len(set(l)) > 1 else "no"}
             for i, l in traslapes.items() if i in ids_muestra]
    return pd.DataFrame(filas, columns=["ID", "Polígonos", "Clases", "Clases distintas"]).sort_values("ID")
 
 
def tabla_pocas_celdas(m, ids_pocas):
    """Si cada evento con pocas celdas está en la muestra y, si está, dónde cae su corona."""
    pc = pd.DataFrame({"ID": sorted(ids_pocas)})
    pc = pc.merge(m[["ID", "GRUPO", "TIPO", "X", "Y"]], on="ID", how="left")
    pc.insert(1, "En la muestra", pc["ID"].isin(m["ID"]).map({True: "sí", False: "no"}))
    pc[["X", "Y"]] = pc[["X", "Y"]].round(1)
    return pc.sort_values(["En la muestra", "ID"], ascending=[False, True])
 
 
def tabla_tipo_grupo(m):
    return pd.crosstab(m["TIPO"].astype(str), m["GRUPO"].astype(str),
                       margins=True, margins_name="Total")
 
 
def tabla_clases(m, col, areas_km2):
    """Eventos por clase y grupo, con el área de la clase en la capa. Incluye
    las clases sin ningún evento (sirve para decidir el agrupamiento)."""
    d = m[[col, "GRUPO"]].copy()
    d[col] = d[col].where(d[col].notna(), "(sin dato)")
    t = pd.crosstab(d[col], d["GRUPO"].astype(str))
    if areas_km2:
        indice = list(t.index) + [c for c in areas_km2 if c not in t.index]
        t = t.reindex(indice, fill_value=0)
    t["Total"] = t.sum(axis=1)
    t["% de la muestra"] = (100 * t["Total"] / t["Total"].sum()).round(1)
    if areas_km2:
        t["Área en la capa (km²)"] = [round(areas_km2[c], 2) if c in areas_km2 else np.nan
                                      for c in t.index]
    t = t.sort_values("Total", ascending=False)
    t.columns.name = None
    t.index.name = "Clase"
    return t.reset_index()
 
 
def medidas_tamano(m):
    """Ancho, largo, semiancho y radio equivalente de cada evento."""
    m = m.copy()
    w = pd.to_numeric(m["ANCHO_R"], errors="coerce")
    l = pd.to_numeric(m["LARGO_R"], errors="coerce")
    m["ANCHO"], m["LARGO"] = np.minimum(w, l), np.maximum(w, l)
    m["SEMIANCHO"] = m["ANCHO"] / 2
    m["RADIO_EQ"] = np.sqrt((10 ** m["LOG_AREA"]) / np.pi)
    return m
 
 
def resumen_tamano(m):
    """Percentiles de las medidas de tamaño, para todos y por grupo."""
    medidas = {
        "Ancho (m)": "ANCHO",
        "Semiancho (m)": "SEMIANCHO",
        "Largo (m)": "LARGO",
        "Radio equivalente raíz(A/pi) (m)": "RADIO_EQ",
    }
    grupos = [("Todos", m)] + [(str(g), d) for g, d in m.groupby("GRUPO")]
    filas = []
    for nombre_g, g in grupos:
        for etiqueta, col in medidas.items():
            s = g[col].dropna()
            fila = {"Grupo": nombre_g, "Medida": etiqueta, "n": len(s)}
            for p, nom in ((10, "P10"), (25, "P25"), (50, "Mediana"), (75, "P75"), (90, "P90")):
                fila[nom] = round(float(np.percentile(s, p)), 1) if len(s) else np.nan
            filas.append(fila)
    return pd.DataFrame(filas)

# Funciones con arcpy


### **Que hace**
Define las funciones que leen las capas y cruzan las coronas con los condicionantes.

Se compone de cinco funciones

|Función|Descripción|
|---|---|
|`revisar_campos(ruta, campos, nombre)`|Detiene la corrida si falta un campo y ista los campos disponibles con su alias|
|`leer(ruta, campos, nombre, tokens)`|lee campos de una capa a una tabla de **pandas** con un **SearchCursor** |
|`es_raster(ruta)`| Identifica las capas de tipo raster |
|`dominio_del_campo(capa, campo)`|En caso de que un campo tenga un dominio  de valores codificados, busca la geodatabase de la capa y devuelve la tabla código → descripción.|
|`aplicar_dominio()`|Se activa en caso de que un campo use un dominio de valores codificados y cambia cada código por su descripción |
|`clase_en_coronas(puntos, campo_id, capa, campo_clase)`|Devuelve la clase de la capa en cada corona, el área de cada clase en $km^{2}$, avisos y las coronas que caen en más de un polígono|


In [67]:

def revisar_campos(ruta, campos, nombre):

    lista = arcpy.ListFields(ruta)
    mayus = {f.name.upper() for f in lista}
    faltan = [c for c in campos if not c or c.upper() not in mayus]
    if faltan:
        disponibles = "\n".join(
            f"    {f.name}" + (f"   (alias: {f.aliasName})" if f.aliasName and f.aliasName != f.name else "")
            for f in lista)
        raise ValueError(f"\n{nombre}: no encuentro los campos {faltan} en\n  {ruta}\n"
                         f"Campos disponibles (escribe el NOMBRE, no el alias):\n{disponibles}\n"
                         "Corrige el nombre en el bloque CONFIGURACIÓN.")
 
 
def leer(ruta, campos, nombre, tokens=()):
    """Lee campos (y tokens como SHAPE@X) de una capa a un DataFrame."""
    revisar_campos(ruta, campos, nombre)
    with arcpy.da.SearchCursor(ruta, list(campos) + list(tokens)) as cur:
        return pd.DataFrame([list(f) for f in cur], columns=list(campos) + list(tokens))
 
 
def es_raster(ruta):
    return arcpy.Describe(ruta).dataType in ("RasterDataset", "RasterLayer", "RasterBand", "MosaicDataset")
 
 
def dominio_del_campo(capa, campo):
    f = next((f for f in arcpy.ListFields(capa) if f.name.upper() == campo.upper()), None)
    if f is None or not f.domain:
        return None, None
    ruta = arcpy.Describe(capa).catalogPath
    pos = ruta.lower().find(".gdb")
    if pos < 0:
        return None, None
    gdb = ruta[:pos + 4]
    for d in arcpy.da.ListDomains(gdb):
        if d.name == f.domain and d.domainType == "CodedValue":
            return d.name, {normalizar_id(k): str(v).strip() for k, v in d.codedValues.items()}
    return None, None
 
 
def clase_en_coronas(puntos, campo_id, capa, campo_clase):
    avisos = []
    if es_raster(capa):
        arcpy.CheckOutExtension("Spatial")
        salida = r"memory\f3s1_valores"
        arcpy.sa.ExtractValuesToPoints(puntos, capa, salida, "NONE", "VALUE_ONLY")
        with arcpy.da.SearchCursor(salida, [campo_id, "RASTERVALU"]) as cur:
            codigos = {normalizar_id(i): codigo_raster(v) for i, v in cur}
        arcpy.management.Delete(salida)
        # Tabla de atributos del ráster: nombre de la clase y área
        cod2clase, areas = {}, {}
        try:
            r = arcpy.Raster(capa)
            km2_celda = r.meanCellWidth * r.meanCellHeight / 1e6
            con_nombre = bool(campo_clase) and campo_clase.upper() != "VALUE"
            campos = ["Value", "Count"] + ([campo_clase] if con_nombre else [])
            with arcpy.da.SearchCursor(capa, campos) as cur:
                for fila in cur:
                    cod = normalizar_id(fila[0])
                    clase = (normalizar_id(fila[2]) or "(sin clase)") if con_nombre else cod
                    cod2clase[cod] = clase
                    areas[clase] = areas.get(clase, 0.0) + fila[1] * km2_celda
        except Exception as e:
            avisos.append(f"No pude leer la tabla de atributos del ráster ({e}); se usan los códigos, sin área.")
            areas = None
        clases = {i: (cod2clase.get(c, c) if c else None) for i, c in codigos.items()}
        return clases, areas, avisos, {}
 
    # Capa de polígonos: unión espacial solo con el ID y la clase
    salida = r"memory\f3s1_union"
    fms = arcpy.FieldMappings()
    for capa_fm, campo, nuevo in ((puntos, campo_id, "ID_U"), (capa, campo_clase, "CLASE_U")):
        fm = arcpy.FieldMap()
        fm.addInputField(capa_fm, campo)
        of = fm.outputField
        of.name, of.aliasName = nuevo, nuevo
        fm.outputField = of
        fms.addFieldMap(fm)
    arcpy.analysis.SpatialJoin(puntos, capa, salida, "JOIN_ONE_TO_MANY", "KEEP_ALL", fms, "INTERSECT")
    por_id = {}
    with arcpy.da.SearchCursor(salida, ["ID_U", "JOIN_FID", "CLASE_U"]) as cur:
        for i, jf, c in cur:
            lista = por_id.setdefault(normalizar_id(i), [])
            if jf is not None and jf >= 0:
                lista.append(normalizar_id(c) or "(sin clase)")
    arcpy.management.Delete(salida)
    clases = {i: (l[0] if l else None) for i, l in por_id.items()}
    traslapes = {i: l for i, l in por_id.items() if len(l) > 1}
 
    areas = {}
    with arcpy.da.SearchCursor(capa, [campo_clase, "SHAPE@AREA"]) as cur:
        for c, a in cur:
            c = normalizar_id(c) or "(sin clase)"
            areas[c] = areas.get(c, 0.0) + (a or 0.0) / 1e6
    if arcpy.Describe(capa).spatialReference.type != "Projected":
        avisos.append("La capa no está proyectada: las áreas no quedan en km².")
 
    nombre_dom, dominio = dominio_del_campo(capa, campo_clase)
    if dominio:
        clases, areas, traslapes = aplicar_dominio(clases, areas, traslapes, dominio)
        avisos.append(f"El campo {campo_clase} usa el dominio '{nombre_dom}': se muestran las "
                      "descripciones en lugar de los códigos.")
    return clases, areas, avisos, traslapes
 

# Diagnóstico principal 

### **¿Qué hace?**

Ejecuta el diagnóstico completo: lee la muestra, la verifica, arma las tablas de tipo, clases y tamaño, y guarda la muestra para el script siguiente.

In [68]:

 
def main():
    if arcpy is None:
        raise RuntimeError("Este script debe correrse en el entorno Python de ArcGIS Pro (no encuentro arcpy).")
    arcpy.env.overwriteOutput = True
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    rep = Reporte(os.path.join(CARPETA_SALIDA, "F3_S1_reporte.txt"))
 
    def sal(nombre):
        return os.path.join(CARPETA_SALIDA, nombre)
 
    try:
        rep.titulo("FASE 3 · SCRIPT 1 — DIAGNÓSTICO DE LA MUESTRA")
        rep(f"Ejecución: {datetime.datetime.now():%Y-%m-%d %H:%M}")
        for nombre, ruta in (("Coronas", CORONAS), ("Rectángulos", RECTANGULOS), ("Geología", GEOLOGIA),
                             ("Geomorfología", GEOMORFOLOGIA), ("Cobertura", COBERTURA)):
            if not arcpy.Exists(ruta):
                raise FileNotFoundError(f"No encuentro la capa {nombre}: {ruta}")
 
        for nombre, ruta, campo in (("Geología", GEOLOGIA, CAMPO_CLASE_GEOLOGIA),
                                    ("Geomorfología", GEOMORFOLOGIA, CAMPO_CLASE_GEOMORFOLOGIA),
                                    ("Cobertura", COBERTURA, CAMPO_CLASE_COBERTURA)):
            if not es_raster(ruta):
                revisar_campos(ruta, [campo], nombre)
 
        # ------------------------------------------------------------------
        # Lectura: coronas (muestra) + rectángulos (medidas de tamaño)
        # ------------------------------------------------------------------
        # Coronas: definen la muestra (ID, grupo, tipo y ubicación)
        campos_c = [CAMPO_ID_CORONAS, CAMPO_GRUPO, CAMPO_TIPO] + ([CAMPO_EXCLUIDO] if CAMPO_EXCLUIDO else [])
        cor = leer(CORONAS, campos_c, "Coronas", ("SHAPE@X", "SHAPE@Y"))
        cor.columns = ["ID", "GRUPO", "TIPO"] + (["EXCLUIDO"] if CAMPO_EXCLUIDO else []) + ["X", "Y"]
        if not CAMPO_EXCLUIDO:
            cor["EXCLUIDO"] = None
        cor["ID"] = cor["ID"].map(normalizar_id)
        cor["GRUPO"] = cor["GRUPO"].map(normalizar_id)
        cor["TIPO"] = cor["TIPO"].map(lambda v: normalizar_id(v) or "(sin tipo)")
        cor["EXCLUIDO"] = cor["EXCLUIDO"].map(normalizar_id)
        n_coronas = len(cor)
        excluidos = cor.loc[cor["EXCLUIDO"].notna(), ["ID", "EXCLUIDO"]]
        cor = cor[cor["EXCLUIDO"].isna()].drop(columns="EXCLUIDO")
 
        # Rectángulos: métricas y tamaño de los 453 eventos
        rec = leer(RECTANGULOS, [CAMPO_ID_RECT, CAMPO_AREA, CAMPO_HL, CAMPO_ELONG, CAMPO_ANCHO, CAMPO_LARGO],
                   "Rectángulos")
        rec.columns = ["ID", "LOG_AREA", "LOG_HL", "LOG_ELONG", "ANCHO_R", "LARGO_R"]
        rec["ID"] = rec["ID"].map(normalizar_id)
        for c, en_log in (("LOG_AREA", AREA_EN_LOG), ("LOG_HL", HL_EN_LOG), ("LOG_ELONG", ELONG_EN_LOG)):
            rec[c] = a_log10(rec[c], en_log)
        n_rect = len(rec)
        dup_rect = int(rec["ID"].duplicated().sum()) + int(rec["ID"].isna().sum())
        rec = rec[rec["ID"].notna()].drop_duplicates("ID")
 
        # Unión que parte de las coronas: quedan solo los 445 de la muestra
        m = cor.merge(rec.assign(_CON_RECT=True), on="ID", how="left")
        m["_CON_RECT"] = m["_CON_RECT"].notna() & m["_CON_RECT"].astype(bool)
 
        # ------------------------------------------------------------------
        # 1. Verificación de la muestra y eventos con pocas celdas
        # ------------------------------------------------------------------
        rep.titulo("1. VERIFICACIÓN DE LA MUESTRA")
        rep("Métricas en log10: " + ", ".join(
            f"{n} ({'ya venía en log' if l else 'calculado aquí'})"
            for n, l in (("área", AREA_EN_LOG), ("H/L", HL_EN_LOG), ("elongación", ELONG_EN_LOG))))
        ids_revision = {normalizar_id(i) for i in IDS_EN_REVISION} - {None}
        rep.tabla(verificar_muestra(m, n_coronas, excluidos, N_ESPERADO, n_rect, dup_rect,
                                    N_RECTANGULOS, ids_revision))
 
        ids_pocas = {normalizar_id(i) for i in IDS_POCAS_CELDAS} - {None}
        rep("\nEventos con pocas celdas:")
        if not ids_pocas:
            rep("  (no se configuraron IDS_POCAS_CELDAS)")
        else:
            t_pc = tabla_pocas_celdas(m, ids_pocas)
            rep.tabla(t_pc)
            t_pc.to_csv(sal("F3_S1_pocas_celdas.csv"), index=False, encoding=ENC)
 
        # ------------------------------------------------------------------
        # 2. Tipo de movimiento × grupo
        # ------------------------------------------------------------------
        rep.titulo("2. TIPO DE MOVIMIENTO × GRUPO (caídas incluidas)")
        t_tg = tabla_tipo_grupo(m)
        rep.tabla(t_tg, index=True)
        t_tg.to_csv(sal("F3_S1_tipo_x_grupo.csv"), encoding=ENC)
        interior = t_tg.drop(index="Total", columns="Total")
        bajas = [f"{i} / {c}: {int(interior.loc[i, c])}"
                 for i in interior.index for c in interior.columns if interior.loc[i, c] < 5]
        if bajas:
            rep.aviso("Combinaciones con menos de 5 eventos (su efecto se estimará con poca precisión): "
                      + "; ".join(bajas))
 
        # ------------------------------------------------------------------
        # 3. Condicionantes categóricos en la corona
        # ------------------------------------------------------------------
        rep.titulo("3. CLASES DE LOS CONDICIONANTES CATEGÓRICOS EN LA CORONA")
        rep("Diagnóstico para agrupar clases. La extracción definitiva (clase mayoritaria "
            "en el buffer) se hace en el script 2.")
        categoricas = {
            "Geología":      (GEOLOGIA, CAMPO_CLASE_GEOLOGIA, "CL_GEOLOGIA"),
            "Geomorfología": (GEOMORFOLOGIA, CAMPO_CLASE_GEOMORFOLOGIA, "CL_GEOMORFO"),
            "Cobertura":     (COBERTURA, CAMPO_CLASE_COBERTURA, "CL_COBERTURA"),
        }
        for nombre, (ruta, campo, col) in categoricas.items():
            rep(f"\n--- {nombre} ---")
            clases, areas, avisos, traslapes = clase_en_coronas(CORONAS, CAMPO_ID_CORONAS, ruta, campo)
            for a in avisos:
                rep.aviso(a)
            t_tr = tabla_traslapes(traslapes, set(m["ID"]))
            if len(t_tr):
                distintas = int((t_tr["Clases distintas"] == "sí").sum())
                rep.aviso(f"{len(t_tr)} coronas caen en más de un polígono; en {distintas} de ellas los "
                          f"polígonos tienen clases distintas. Por ahora se tomó el primero. "
                          f"Detalle en F3_S1_traslapes_{col.lower()}.csv")
                t_tr.to_csv(sal(f"F3_S1_traslapes_{col.lower()}.csv"), index=False, encoding=ENC)
            m[col] = m["ID"].map(lambda i: clases.get(i))
            t_cl = tabla_clases(m, col, areas)
            rep.tabla(t_cl)
            t_cl.to_csv(sal(f"F3_S1_clases_{col.lower()}.csv"), index=False, encoding=ENC)
            reales = t_cl[t_cl["Clase"] != "(sin dato)"]
            rep(f"Clases en la capa: {len(areas) if areas else 'n/d'} | con al menos un evento: "
                f"{int((reales['Total'] > 0).sum())} | con 1 a 4 eventos: "
                f"{int(((reales['Total'] > 0) & (reales['Total'] < 5)).sum())}")
            sin_dato = int(m[col].isna().sum())
            if sin_dato:
                rep.aviso(f"{sin_dato} coronas quedan fuera de esta capa (sin dato).")
 
        # ------------------------------------------------------------------
        # 4. Tamaño de los eventos
        # ------------------------------------------------------------------
        rep.titulo("4. TAMAÑO DE LOS EVENTOS (para fijar el radio del buffer)")
        m = medidas_tamano(m)
        rep(f"Ancho y largo: campos {CAMPO_ANCHO} y {CAMPO_LARGO} de la capa de rectángulos.")
        t_tam = resumen_tamano(m)
        rep.tabla(t_tam)
        t_tam.to_csv(sal("F3_S1_tamano_eventos.csv"), index=False, encoding=ENC)
 
        # ------------------------------------------------------------------
        # Salida: muestra para el siguiente script
        # ------------------------------------------------------------------
        cols = (["ID", "GRUPO", "TIPO", "X", "Y", "LOG_AREA", "LOG_HL", "LOG_ELONG"]
                + [col for _, _, col in categoricas.values()]
                + ["ANCHO", "LARGO", "RADIO_EQ"])
        m[cols].to_csv(sal("F3_S1_muestra.csv"), index=False, encoding=ENC)
        rep.titulo("SALIDAS")
        rep(f"Muestra (n = {len(m)}): F3_S1_muestra.csv. Las columnas CL_* son la clase en la "
            "corona; el script 2 las reemplaza por la clase mayoritaria en el buffer.")
        rep(f"Todo en: {CARPETA_SALIDA}")
 
    finally:
        rep.guardar()
        try:
            arcpy.management.Delete("memory")
        except Exception:
            pass
 
 
if __name__ == "__main__":
    main()

﻿
FASE 3 · SCRIPT 1 — DIAGNÓSTICO DE LA MUESTRA
Ejecución: 2026-10-09 22:02
﻿
1. VERIFICACIÓN DE LA MUESTRA
Métricas en log10: área (ya venía en log), H/L (ya venía en log), elongación (ya venía en log)
                                            Verificación                          Resultado  Estado
                           Eventos en la capa de coronas                                442    info
                                      Excluidos (motivo)                                  0    info
                                   Eventos en la muestra                442 (esperados 445) REVISAR
                    IDs vacíos o repetidos en la muestra                                  0      OK
                       Eventos en la capa de rectángulos                453 (esperados 453)      OK
                   IDs vacíos o repetidos en rectángulos                                  0      OK
                 Eventos de la muestra sin su rectángulo                                  0      